# AfroRisk — exploration des données

Ce notebook reprend le script `python -m afrorisk.eda`. Il charge Home Credit si `data/raw/application_train.csv` est présent, sinon la fixture synthétique. Les chiffres affichés sont ceux du jeu réellement chargé : rien n'est recopié d'un entraînement antérieur.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from afrorisk.eda import build_summary, save_figures
from afrorisk.features import add_application_features
from afrorisk.io import load_tables

tables, source = load_tables()
application = tables["application"]
print(source, application.shape)


## Structure, manquants, doublons, cible

`DAYS_EMPLOYED = 365243` est une sentinelle (souvent les retraités), pas une ancienneté. Elle est traitée comme manquante dans les variables du modèle.


In [ ]:
summary = build_summary(application)
summary


In [ ]:
application.dtypes


In [ ]:
featured = add_application_features(application)
featured["TARGET"] = application["TARGET"].to_numpy()
featured.groupby("TARGET")[["AGE_YEARS", "AMT_INCOME_TOTAL", "CREDIT_INCOME_RATIO", "EXT_SOURCE_2", "EXT_SOURCE_3"]].median(numeric_only=True)


## Figures

Les fichiers sont écrits dans `reports/figures/`. Sur la fixture, les écarts reflètent le signal synthétique (scores externes et retards), pas une population réelle.


In [ ]:
save_figures(application, source)
print("figures enregistrées")


## Lecture

Regarder d'abord le taux de défaut et la part de manquants sur `EXT_SOURCE_1`, `EXT_SOURCE_3` et `OCCUPATION_TYPE`. Ensuite comparer les médianes d'âge, de ratio crédit/revenu et de scores externes entre `TARGET = 0` et `TARGET = 1`. Une médiane de score plus basse chez les défauts va dans le sens attendu sur Home Credit ; sur la fixture, c'est le signal construit pour les tests.

Les tables secondaires ne sont pas détaillées ici ligne à ligne. Leurs agrégations (retards, refus, utilisation de carte) sont dans `afrorisk.features` et n'utilisent pas `TARGET`.
